# SHAP Explainability Analysis

This notebook explains the predictions of the trained LightGBM
late-delivery-risk model.

The goal is to identify:
- Which features influence late-delivery predictions
- Which features increase or decrease predicted risk
- Which features are most important overall
- How individual shipments receive their predictions

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
from pathlib import Path

In [4]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "outputs" / "APL_Logistics_ml_ready.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "lightgbm_delay_risk_model.joblib"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Model path:", MODEL_PATH)

Project root: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project
Data path: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\outputs\APL_Logistics_ml_ready.csv
Model path: c:\Users\ganesh\Desktop\All Projects\Build Logistic Analysis Project\models\lightgbm_delay_risk_model.joblib


In [5]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (180519, 37)


,Type,Days for shipment (scheduled),Benefit per order,Sales per customer,Category Id,Category Name,Customer City,Customer Country,Customer Segment,Customer State,...,Product Name,Product Price,Shipping Mode,discount_amount_per_unit,sales_per_quantity,profit_per_quantity,price_discount_interaction,scheduled_days_bucket,geo_market_region,Late_delivery_risk
0,DEBIT,4,159.69,472.45,9,Cardio Equipment,Brownsville,EE. UU.,Consumer,TX,...,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class,5.500,99.99,31.938,5.9994,4,Pacific Asia | South Asia,1
1,DEBIT,4,48.71,167.96,29,Shop By Sport,Littleton,EE. UU.,Consumer,CO,...,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,6.398,39.99,9.742,6.3984,4,LATAM | Central America,0
2,DEBIT,4,87.36,181.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,18.000,199.99,87.360,17.9991,4,LATAM | Central America,0
3,DEBIT,4,-41.89,175.99,48,Water Sports,Littleton,EE. UU.,Consumer,CO,...,Pelican Sunstream 100 Kayak,199.99,Standard Class,24.000,199.99,-41.890,23.9988,4,USCA | East of USA,1
4,DEBIT,4,10.00,40.00,24,Women's Apparel,Littleton,EE. UU.,Consumer,CO,...,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class,10.000,50.00,10.000,10.0000,4,USCA | East of USA,1


In [6]:
lightgbm_model = joblib.load(MODEL_PATH)

print("LightGBM model loaded successfully.")
print("Number of model features:", len(lightgbm_model.feature_name_))

LightGBM model loaded successfully.
Number of model features: 33


In [8]:
TIMING_SENSITIVE_COLUMNS = [
    "Order Profit Per Order",
    "profit_per_quantity",
    "Benefit per order"
]

TARGET = "Late_delivery_risk"

x = df.drop(columns=[TARGET], errors="ignore").copy()

x = x.drop(
    columns=TIMING_SENSITIVE_COLUMNS,
    errors="ignore"
)

y = df[TARGET].astype(int)

print("Feature matrix shape:", x.shape)
print("Target shape:", y.shape)

Feature matrix shape: (180519, 33)
Target shape: (180519,)


In [9]:
evaluation_columns = x.columns.tolist()

feature_mapping = {}

for col in evaluation_columns:
    lightgbm_name = col.replace(" ", "_")
    feature_mapping[lightgbm_name] = col

model_features = lightgbm_model.feature_name_

matched_columns = []

for model_feature in model_features:
    if model_feature in feature_mapping:
        matched_columns.append(
            feature_mapping[model_feature]
        )

print("Model features:", len(model_features))
print("Matched columns:", len(matched_columns))

Model features: 33
Matched columns: 33


In [10]:
x_shap = x[
    matched_columns
].copy()

print("SHAP datadet shape:", x_shap.shape)

SHAP datadet shape: (180519, 33)


In [11]:
x_shap_sample = x_shap.sample(
    n = min(5000, len(x_shap))
)

print("SHAP sample shape:", x_shap_sample.shape)

SHAP sample shape: (5000, 33)
